# 🧠 Building Your First Local AI Agent (LangGraph + Ollama)
A fundamentals-first notebook covering agents, tools, memory, and design patterns.

## 🚀 Getting Started
This notebook teaches how to build a **local AI agent** using:
- Ollama (Qwen3)
- LangGraph
- OpenAI-compatible client


In [ ]:
%pip install openai langgraph

## 🧩 What is an AI Agent?
An agent is:
1. A model (LLM)
2. Tools it can use
3. Memory (state)
4. A loop (decide → act → observe)


## 🧱 Core Components of Agents
- LLM (reasoning engine)
- Tools (functions/API calls)
- Memory (state/messages)
- Router (decides next step)
- Execution loop (LangGraph)


## 🔁 Agent vs LLM Pipeline
| Pipeline | Agent |
|----------|------|
| Fixed steps | Dynamic decisions |
| No memory | Stateful memory |
| One-pass | Multi-step reasoning |


## 🧠 Types of Agents
- Reactive Agents (simple tool use)
- Planner-Executor Agents
- ReAct Agents (Reason + Act)
- Multi-agent systems


## 🏗️ Agent Design Patterns
- Router Pattern
- Tool-Calling Pattern
- Planner → Executor Pattern
- Reflection Pattern


## 🧠 Structured Reasoning
### Chain of Thought (CoT)
- Step-by-step reasoning

### Tree of Thoughts (ToT)
- Explore multiple reasoning paths


## 🔧 Tools Example
We define simple Python functions as tools.

In [ ]:

def add(a, b):
    return a + b

def multiply(a, b):
    return a * b

def weather(city):
    return f"Sunny in {city}"


## 🔄 LangGraph Agent Loop
LLM → Tool → LLM → Tool → Final Answer

In [ ]:

from openai import OpenAI
from langgraph.graph import StateGraph, END
import json

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

TOOLS = {
    "add": add,
    "multiply": multiply,
    "weather": weather
}

class State(dict):
    pass


## 🤖 LLM Node (Decision Maker)

In [ ]:

def llm_node(state):
    resp = client.chat.completions.create(
        model="qwen3:1.7b",
        messages=state["messages"],
        tool_choice="auto"
    )
    msg = resp.choices[0].message
    return {"messages": state["messages"] + [msg]}


## 🛠️ Tool Node (Execution Layer)

In [ ]:

def tool_node(state):
    msg = state["messages"][-1]
    if not msg.tool_calls:
        return END

    call = msg.tool_calls[0]
    name = call.function.name
    args = json.loads(call.function.arguments)

    result = TOOLS[name](**args)

    tool_msg = {
        "role": "tool",
        "tool_call_id": call.id,
        "content": str(result)
    }

    return {"messages": state["messages"] + [tool_msg]}


## 🌐 Build Graph

In [ ]:

graph = StateGraph(State)
graph.add_node("llm", llm_node)
graph.add_node("tool", tool_node)

graph.set_entry_point("llm")
graph.add_edge("llm", "tool")
graph.add_edge("tool", "llm")

agent = graph.compile()


## ▶️ Run Agent

In [ ]:

def run(query):
    state = {"messages": [{"role": "user", "content": query}]}
    return agent.invoke(state)["messages"][-1]["content"]

print(run("Add 5 and 10"))
print(run("What is weather in Dubai?"))
